<a href="https://colab.research.google.com/github/AdrianDVnqn/UA_MDM_Labo2_Grupo12/blob/LightGBM_Test/tutoriales/06_text_classification_optuna_VF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fuentes: https://medium.com/nlplanet/fine-tuning-distilbert-on-senator-tweets-a6f2425ca50e

#### **Instalar Modulos**

conda install datasets=="2.20.0"

conda install transformers=="4.40.1"

conda install numpy=="1.26.4" # La última versión no funciona bien


Como aclaración, en el caso de correr este script en Colab hay que ejectutar la siguiente celda y luego reiniciar la sesión, ya que de esa manera se actualiza numpy

In [ ]:
! pip uninstall -y numpy
! pip install numpy==1.26.4

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 102.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


Una vez reiniciada la sesión se controla que la versión de numpy sea la correcta para poder correr las lineas de código que forman parte del presente notebook

In [ ]:
import numpy as np
print(np.__version__)

1.26.4


In [ ]:
!pip install datasets==2.20.0
!pip install transformers==4.40.1

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x

In [ ]:
#Importación de la libreria optuna
!pip install optuna
!pip install -U kaleido
!pip install optuna-dashboard
#!pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 10.2 MB/s eta 0:00:00


In [ ]:
####IMPORTANTE CARGAR UTILS.PY
from google.colab import files
uploaded = files.upload()

Saving utils.py to utils.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Data processing
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import copy

import time
import datetime

from sklearn.metrics import confusion_matrix, cohen_kappa_score

from datasets import Dataset,  DatasetDict

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding, AutoModelForSequenceClassification, AdamW, get_scheduler

# Progress bar
from tqdm.auto import tqdm

from utils import plot_confusion_matrix, get_artifact_filename

from joblib import load, dump

# Verificamos que CUDA está funcional
torch.cuda.is_available()

True

**Bajamos el modelo**

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

**Armado de los Datasets**

El código lo vinculamos a una carpeta de Drive para directamente tomar los archivos desde la nube a fin de no tener que subir archivos y luego descargarlos. Es decir, buscamos automatizar la carga de la información así como el guardado de los archivos que se generan al entrenar el modelo.

In [ ]:
# Paths
# Definir la ruta base para tu Google Drive
BASE_DIR_DRIVE = '/content/drive/MyDrive'
PATH_TO_TRAIN = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/input/petfinder-adoption-prediction/train/train_final_thres.csv")
# Artefactos a subir a optuna
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_temp_artifacts")

# Artefactos que optuna gestiona
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_artifacts")
PATH_TO_DB = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/db_cv_10052025_2.sqlite3")

# Parametros y variables
SEED = 42
TEST_SIZE = 0.2

BATCH_SIZE = 64

MODEL_NAME = '06 Bert_1'

MODEL_VERSION = '1.1'

In [ ]:
# Cargar los datos
df = pd.read_csv(PATH_TO_TRAIN)
df = df[df['Description'].notnull()]
df['labels'] = df["AdoptionSpeed"]

# Dividir los datos usando sklearn
#train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df.AdoptionSpeed)

# study_lgb = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name="04 - LGB Multiclass CV",
#                            load_if_exists = True)

study_lgb = optuna.create_study(direction='maximize',
                            storage=f"sqlite:///{PATH_TO_DB}",  # Specify the storage URL here.
                            study_name="04 - LGB Multiclass CV 1052025",
                           load_if_exists = True)

lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

train_df = df[~df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)
test_df = df[df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)

# Convertir a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Combinar en un DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'val': test_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Hacer una lista de columnas para remover antes de la tokenización
cols_to_remove = [col for col in dataset["train"].column_names if col != 'labels']
print(cols_to_remove)

[I 2025-05-11 14:08:17,782] Using an existing study with name '04 - LGB Multiclass CV 1052025' instead of creating a new one.


Stringifying the column:   0%|          | 0/9586 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/9586 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/2398 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/2398 [00:00<?, ? examples/s]

['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', 'rescuer_ratio_speed_0_rescuer_ratio', 'rescuer_ratio_speed_1_rescuer_ratio', 'rescuer_ratio_speed_2_rescuer_ratio', 'rescuer_ratio_speed_3_rescuer_ratio', 'rescuer_ratio_speed_4_rescuer_ratio', 'Quantity_Total', 'Color', 'Tiene_Nombre', 'Es_Gratis', 'Age_Fee', 'Description_limpia', 'longitud_descripcion', 'cantidad_palabras_no_stopwords', 'stopwords_eliminadas', 'descripcion_para_analisis', 'Nombres_limpios', 'categoria_rescatista', 'cantidad_animales', 'disponibilidad_imagen', 'estado_sanitario', 'AgeCategory', 'State_importance']


In [ ]:
# Tokenize and encode the dataset
def tokenize(batch):
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokenized_batch = tokenizer(batch["Description"], padding=True, truncation=True, max_length=512)
    return tokenized_batch

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=4)

# Set dataset format for PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check the output
print(dataset_enc["train"].column_names)



Map (num_proc=4):   0%|          | 0/9586 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in ver

Map (num_proc=4):   0%|          | 0/2398 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in ver

['labels', 'input_ids', 'attention_mask']


In [ ]:
# Instantiate a data collator with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create data loaders for to reshape data for PyTorch model
train_dataloader = DataLoader(
    dataset_enc["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc["val"], batch_size=BATCH_SIZE, collate_fn=data_collator
)

In [ ]:
test_sample_ids =[i for i in test_df.PetID]

In [ ]:
# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                           num_labels=num_labels)

Number of labels: 5


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

# Set the device automatically (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Move model to device
model.to(device)

cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
!pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 96.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


In [ ]:
def train_val(model, dataloaders, datasets, device, num_epochs=4, lr=0.001, trial=None):

    since = time.time()

    # Create the optimizer
    optimizer = AdamW(model.parameters(), lr=lr)

    # Further define learning rate scheduler
    num_training_batches = len(train_dataloader)
    num_training_steps = num_epochs * num_training_batches
    lr_scheduler = get_scheduler(
        "linear",                   # linear decay
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )


    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999

    train_losses = []
    val_losses = []

    try:
        previous_best = study.best_value
    except:
        previous_best = -999


    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        kappa_labels_true = []
        kappa_labels_predicted = []
        output_scores = []

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for batch in tqdm(dataloaders[phase]):
                batch = batch.to(device)
                #inputs = inputs.to(device)
                labels = batch.labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(**batch)
                    loss = outputs.loss

                    preds = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    preds_labels = torch.argmax(preds, dim=-1)


                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                    elif phase == 'val':
                        kappa_labels_true.extend(labels.cpu().numpy().tolist())
                        kappa_labels_predicted.extend(preds_labels.cpu().numpy().tolist())
                        outputs_np = preds.cpu().numpy()
                        output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                # Statistics
                running_loss += loss.item() * labels.size(0)
                running_corrects += torch.sum(preds_labels == labels.data)

                #END OF BATCH

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc = running_corrects.double() / len(datasets[phase])

            if phase == 'train':
                train_losses.append(epoch_loss)
                kappa_score = np.nan
            else:
                val_losses.append(epoch_loss)
                kappa_score = cohen_kappa_score(kappa_labels_true,
                                  kappa_labels_predicted,
                                  weights = 'quadratic')



            print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {kappa_score:.3f}')

            # If this is the best Epoch so far -> Deep copy the model
            if phase == 'val' and kappa_score > best_kappa:
                best_acc = epoch_acc
                best_kappa = kappa_score
                best_model_wts = copy.deepcopy(model.state_dict())


                #Best Epoch within a trial and better than previous trials
                if trial is not None and best_kappa > previous_best:

                    #Save test dataset with predictions
                    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                    predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                'pred':output_scores}).merge(test_df, on='PetID')
                    dump(predicted_df, predicted_filename)

                    #Generate and save CM
                    cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                    plot_confusion_matrix(kappa_labels_true,kappa_labels_predicted).write_image(cm_filename)

            #END OF PHASE

        #END OF EPOCH

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:.2f}%'.format(best_acc * 100))

    # Load best model weights
    model.load_state_dict(best_model_wts)

    # Save in optuna trial the best test dataset, cm and model weights
    if trial is not None and best_kappa > previous_best:
        upload_artifact(trial, predicted_filename, artifact_store)

        upload_artifact(trial, cm_filename, artifact_store)

        file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
        model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
        torch.save(model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
        upload_artifact(trial, model_path, artifact_store)

    return model,best_kappa



In [ ]:

# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")


Number of labels: 5


In [ ]:
best_model,_ = train_val(model,
                       dataloaders={'train': train_dataloader,
                                    'val': eval_dataloader},
                       datasets=dataset_enc,
                       device=device,
                       lr = 5e-5,
                       num_epochs=15)


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0/14
----------


  0%|          | 0/150 [00:00<?, ?it/s]

Train Loss: 1.4517 Acc: 31.61% Kappa: nan


  0%|          | 0/38 [00:00<?, ?it/s]

...

Val Loss: 3.3278 Acc: 37.91% Kappa: 0.271
Epoch 14/14
----------


  0%|          | 0/150 [00:00<?, ?it/s]

Train Loss: 0.1105 Acc: 95.71% Kappa: nan


  0%|          | 0/38 [00:00<?, ?it/s]

Val Loss: 3.2593 Acc: 37.11% Kappa: 0.274
Training complete in 27m 45s
Best val Acc: 36.36%


In [ ]:
# Guardo el modelo
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
print(f'Modelo guardado en {model_path}')

Modelo guardado en /content/drive/MyDrive/Colab Notebooks/LABO_II/work/optuna_temp_artifacts/06 Bert_1_1.1_20250511_143647.pth


In [ ]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def optuna_train(trial):

    epochs = trial.suggest_int('epochs', 1, 2)

    lr = trial.suggest_float('lr', 0.00001, 0.0001, log=True)

    _,best_score = train_val(model,
                       dataloaders={'train': train_dataloader,
                                    'val': eval_dataloader},
                       datasets=dataset_enc,
                       device=device,
                       num_epochs=epochs,
                       lr=lr,
                       trial=trial)


    return(best_score)

In [ ]:
PATH_TO_DB = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/db_bert_11052025.sqlite3")
study = optuna.create_study(direction='maximize',
                            storage=f"sqlite:///{PATH_TO_DB}",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)
study.optimize(optuna_train, n_trials=10)

[I 2025-05-11 14:37:00,638] A new study created in RDB with name: 06 Bert_1_1.1


Epoch 0/1
----------


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/150 [00:00<?, ?it/s]

Train Loss: 0.1196 Acc: 95.43% Kappa: nan


...

  0%|          | 0/150 [00:00<?, ?it/s]

Train Loss: 0.1323 Acc: 94.59% Kappa: nan


  0%|          | 0/38 [00:00<?, ?it/s]

[I 2025-05-11 15:08:29,899] Trial 9 finished with value: 0.26431530285596516 and parameters: {'epochs': 2, 'lr': 9.309875204241658e-05}. Best is trial 7 with value: 0.2901392843390067.


Val Loss: 3.5800 Acc: 36.53% Kappa: 0.255
Training complete in 3m 42s
Best val Acc: 36.86%


In [ ]:
#ESTA CELDA HAY QUE ELIMINARLA
#Linea de código para cargar el LightGBM que luego voy a utilizar en el blend.
#Todo esto se hace para comparar los datasets y que esté trabajando con los mismos dato.s
bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study,'test')))
bert_dataset['pred'] = [np.zeros(5) if type(i) is float else  i for i in bert_dataset['pred'] ]
bert_dataset['pred'] = [r.argmax() for r in bert_dataset['pred']]